# **walter**

## **Project Setup**

In [1]:
import sys
import os
from pathlib import Path
import subprocess
import getpass

IN_COLAB = "google.colab" in sys.modules

REPO_NAME = "walter"
GIT_BRANCH = "main"
REPO_PATH = Path("/content") / REPO_NAME


def get_tokens():
    hf_token = os.getenv("HF_TOKEN") or getpass.getpass("HF token: ")
    github_token = os.getenv("GITHUB_TOKEN") or getpass.getpass("GitHub token: ")
    return hf_token, github_token


def install_core_ml_stack():
    # Only ML libraries (safe layer)
    subprocess.run([
        sys.executable, "-m", "pip", "install",
        "transformers==4.44.2",
        "accelerate==0.33.0"
    ], check=True)

def setup_repo(github_token):
    os.chdir("/content")

    repo_url = f"https://{github_token}@github.com/Mango-Cats/{REPO_NAME}.git"

    if REPO_PATH.exists():
        os.chdir(REPO_PATH)
        subprocess.run(["git", "fetch", "origin"], check=True)
        subprocess.run(["git", "reset", "--hard", f"origin/{GIT_BRANCH}"], check=True)
    else:
        subprocess.run(["git", "clone", repo_url], check=True)
        os.chdir(REPO_PATH)

    # Install project AFTER HF stack is stable
    subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], check=True)


if IN_COLAB:
    print("Colab detected")

    hf_token, github_token = get_tokens()

    install_core_ml_stack()
    setup_repo(github_token)

    os.chdir(REPO_PATH)

from huggingface_hub import HfApi

api = HfApi(token=hf_token)

print(api.whoami())


import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Project root:", REPO_PATH)
print("Torch:", torch.__version__)
print("Device:", DEVICE)

Colab detected
{'type': 'user', 'id': '68b91f73b81aa6dfbd75984b', 'name': 'zrygan', 'fullname': 'Zhean Robby Ganituen', 'email': 'zhean_robby_ganituen@dlsu.edu.ph', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1780272000, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/no-auth/Fp4ErHoZ_gN2BtQvG9Iff.png', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'walter', 'role': 'read', 'createdAt': '2026-05-02T02:47:29.965Z'}}}
Project root: /content/walter
Torch: 2.10.0+cu128
Device: cuda


## **Nomenclature and Terminologies**

The dataset $\mathcal{D}_{\text{raw}}$ (represented as `D_raw` in the source code) refers to the raw Philippine human-drug registry, which is freely available as a `.csv` file at [https://verification.fda.gov.ph/drug_productslist.php](https://verification.fda.gov.ph/drug_productslist.php).

The intermediate dataset, $\mathcal{D}_{\text{clean}}$ (`D_clean`), is the result of passing $\mathcal{D}_{\text{raw}}$ through the preprocessing pipeline. 

The final dataset, $\mathcal{D}_{\text{train}}$ (`D_train`), is used to train a weighted sum of similarity measures via a genetic algorithm. It consists of ordered pairs of drugs formed from the cleaned registry, such that every pair $(x, y) \in \mathcal{D}_{\text{clean}} \times \mathcal{D}_{\text{clean}}$. This training dataset is partitioned into two disjoint subsets:

* **$P \subset \mathcal{D}_{\text{train}}$** (`P`) is the set of known positives, consisting of ordered drug pairs that are manually verified as LASA.
* **$U \subset \mathcal{D}_{\text{train}}$** (`U`) is the unlabeled noise set, consisting of randomly paired drugs from $\mathcal{D}_{\text{clean}}$. $U$ acts as the noise class (its true labels are unknown), so it may contain undetected LASA pairs.

Furthermore, $|U| \gg |P|$, $P \cap U = \emptyset$, and $P \cup U = \mathcal{D}_{\text{train}}$.

## **Preprocessing**

The first step is to preprocess (load, validate, clean) the FDA human drug registry dataset to construct our $\mathcal{D}_\text{clean}$ dataset.

Ensure that the FDA human drug registry dataset exists anywhere starting from the root folder and has the same filename defined by `PRIMARY_FNAME`.

The code for this section is located at [`/src/preprocessing.py`](/src/preprocessing.py).

The function `master_maker` is the coordinator function that performs data loading, validation, cleaning, and reporting. 

In [6]:
import pandas as pd
import src.preprocessing as pre
from src.utils import finder
skip = True

if skip:
    filename = finder("cleaned_drug_products.csv")
    D_clean = pd.read_csv(filename)
else:
    D_clean = pre.master_maker(sort=True, save=True)

Let's look at the info of the dataset.

In [7]:
D_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22838 entries, 0 to 22837
Data columns (total 1 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   Brand Name  22838 non-null  object
dtypes: object(1)
memory usage: 178.6+ KB


Then, the head of the dataset.

In [8]:
D_clean.head()

,Brand Name
0,0.9% NaCl-Sapher
1,0.9% Sodchlorsaph
2,1 Ceeplus
3,1000Vc
4,2-Gen


Finally, let's look at a slice of 10 entries in the dataset by using the `get_rand_entries()` function.

In [9]:
display(pre.get_rand_entries(df=D_clean, count=10))

,Brand Name
6986,Enhancin
6987,Enhanzee
6988,Enhanzee Plus
6989,Enhertu
6990,Enlight
6991,Ennazole
6992,Enocee
6993,Enocee Plus
6994,Enoclex
6995,Enocort


## **True LASA Pairs**

Now that we have the $\mathcal{D}_\text{clean}$ we can now proceed with constructing $\mathcal{D}_\text{train}$. We will prioritize constructing the subset of true LASA pairs, or the set $P$.

The code for this section is located at [`/src/proposer/`](/src/proposer/).

### **HuggingFace Models**

In [10]:
from src.proposer.hf import HFModel, response
from src.proposer.prompt import construct_user_prompt
from src.preprocessing import table_to_string

model = HFModel.DEEPSEEK

iters = 1
P_hf = []

for _ in range(iters):
    sample_df = D_clean.sample(n=1)
    remaining_df = D_clean.drop(sample_df.index)

    sample_str = table_to_string(sample_df)
    remaining_str = table_to_string(remaining_df)
    
    user_prompt = construct_user_prompt(sample_str,remaining_str, 5)

    proposals = response(user_prompt, model=model)

    P_hf.append((sample_str, proposals))

print(P_hf)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


TypeError: LlamaForCausalLM.__init__() got an unexpected keyword argument 'dtype'

## **Noise Pairs**

Now that we have $P$, we can now complete constructing $\mathcal{D}_\text{train}$ by constructing the set $U$ or the unlabeled noise set.

The code for this section is located at [`/src/noise.py`](/src/noise.py)

In [ ]:
import src.noise as noise
from src.utils import finder

sample_file: Path = finder(fname="sample_true_lasa.csv")
sample_P: pd.DataFrame = pd.read_csv(filepath_or_buffer=sample_file)

lasa_set = noise.get_lasa_set(true_df=sample_P)

assert len(lasa_set) == 20

sample_U = noise.make_noise(fda_df=D_clean, true_df=sample_P, n=3)

sample_U

## **Assembling**

Now that both subsets are complete. Assembling $\mathcal{D}_\text{train}$ is simply a concatenation of $P$ and $U$. 

In [ ]:
...